In [5]:
!pip install pandas

/media/E/term6/DS/Datasets/task 3 - datasets/.venv/bin/pip: 2: exec: /home/sajjad/Downloads/Datasets/task 3 - datasets/.venv/bin/python: not found


In [6]:
import pandas as pd

jobs = pd.read_csv('jobs.csv')
candidates = pd.read_csv('candidates.csv')
applications_train = pd.read_csv('applications_train.csv')

print("--- 1. Unique Values ---")
print("Currencies:", jobs['salary_currency'].dropna().unique())
print("English Levels:", candidates['english_proficiency'].dropna().unique())
print("Education Levels:", candidates['education_level'].dropna().unique())
print("\n")

print("--- 2. Missing Values ---")
print("[Jobs]\n", jobs.isnull().sum()[jobs.isnull().sum() > 0])
print("\n[Candidates]\n", candidates.isnull().sum()[candidates.isnull().sum() > 0])
print("\n[Train]\n", applications_train.isnull().sum()[applications_train.isnull().sum() > 0])
print("\n")

print("--- 3. Data Types ---")
print("[Jobs]\n", jobs.dtypes)
print("\n[Candidates]\n", candidates.dtypes)
print("\n[Train]\n", applications_train.dtypes)
print("\n")

print("--- 4. Label Distribution ---")
print(applications_train['relevance_label'].value_counts().sort_index())

--- 1. Unique Values ---
Currencies: <StringArray>
[  'USD',   'EUR',   'IRR', 'IRR  ',  'USD ',   'irr', 'USD  ',   'usd',
   'eur',  'EUR ',  'IRR ', 'EUR  ']
Length: 12, dtype: str
English Levels: <StringArray>
['Upper-Intermediate',                 'C2',                 'C1',
                 'B1',                 'B2',                 'A2',
             'Native',                 'A1',           'Beginner',
           'Advanced',       'Intermediate',         'Elementary',
             'Fluent',          'Bilingual']
Length: 14, dtype: str
Education Levels: <StringArray>
[        'PhD',    'Bachelor',      'Master', 'High School',        'B.A.',
        'B.S.',         'BSc',        'M.S.',        'M.A.',       'Ph.D.',
         'MSc',          'HS',   'Doctorate',   'Secondary']
Length: 14, dtype: str


--- 2. Missing Values ---
[Jobs]
 required_skills         248
min_years_experience    105
max_years_experience    495
remote_allowed          265
company_size            377
dtype:

In [7]:

jobs = pd.read_csv('jobs.csv')
candidates = pd.read_csv('candidates.csv')
applications_train = pd.read_csv('applications_train.csv')

print("--- 1. Applicants per Job ---")
applicants_count = applications_train.groupby('job_id').size()
print("Min applicants:", applicants_count.min())
print("Max applicants:", applicants_count.max())
print("Mean applicants:", applicants_count.mean())

print("\n--- 2. Date Ranges ---")
print("Job Posted Min/Max:", jobs['job_posted_date'].min(), "-", jobs['job_posted_date'].max())
print("Application Min/Max:", applications_train['application_date'].min(), "-", applications_train['application_date'].max())

print("\n--- 3. Top Locations ---")
print("Jobs Top 10:\n", jobs['job_location'].value_counts().head(10))
print("Candidates Top 10:\n", candidates['candidate_location'].value_counts().head(10))

--- 1. Applicants per Job ---
Min applicants: 1
Max applicants: 150
Mean applicants: 32.38941914371421

--- 2. Date Ranges ---
Job Posted Min/Max: 01/04/2023 - 31/01/2023
Application Min/Max: 2023-01-01 - 2024-06-30

--- 3. Top Locations ---
Jobs Top 10:
 job_location
Tabriz        225
Madrid        187
Singapore     184
Tokyo         182
Vancouver     181
Mashhad       180
Isfahan       180
Manchester    180
Toronto       179
Qom           178
Name: count, dtype: int64
Candidates Top 10:
 candidate_location
Dubai         1804
Amsterdam     1801
Boston        1796
Singapore     1788
Toronto       1784
Manchester    1778
Karaj         1760
Berlin        1748
Qom           1743
Paris         1742
Name: count, dtype: int64


In [8]:
import pandas as pd
import numpy as np

def run_phase_one_two(train_path, test_path, jobs_path, candidates_path):
    df_train = pd.read_csv(train_path)
    df_test = pd.read_csv(test_path)
    jobs = pd.read_csv(jobs_path)
    candidates = pd.read_csv(candidates_path)

    jobs['salary_currency'] = jobs['salary_currency'].astype(str).str.strip().str.upper()
    exchange_rates = {'USD': 1.0, 'EUR': 1.08, 'IRR': 0.000002}
    
    jobs['salary_min_usd'] = jobs.apply(lambda row: row['salary_min'] * exchange_rates.get(row['salary_currency'], 1.0) if pd.notnull(row['salary_min']) else np.nan, axis=1)
    jobs['salary_max_usd'] = jobs.apply(lambda row: row['salary_max'] * exchange_rates.get(row['salary_currency'], 1.0) if pd.notnull(row['salary_max']) else np.nan, axis=1)

    candidates['english_proficiency'] = candidates['english_proficiency'].astype(str).str.strip()
    eng_map = {'A1': 1, 'Beginner': 1, 'Elementary': 1, 'A2': 2, 'Pre-Intermediate': 2, 'B1': 3, 'Intermediate': 3, 'B2': 4, 'Upper-Intermediate': 4, 'C1': 5, 'Advanced': 5, 'C2': 6, 'Fluent': 6, 'Bilingual': 6, 'Native': 6}
    candidates['english_score'] = candidates['english_proficiency'].map(eng_map)

    candidates['education_level'] = candidates['education_level'].astype(str).str.strip()
    edu_map = {'High School': 1, 'HS': 1, 'Secondary': 1, 'Bachelor': 2, 'B.A.': 2, 'B.S.': 2, 'BSc': 2, 'Master': 3, 'M.S.': 3, 'M.A.': 3, 'MSc': 3, 'PhD': 4, 'Ph.D.': 4, 'Doctorate': 4}
    candidates['edu_score'] = candidates['education_level'].map(edu_map)

    jobs['job_location'] = jobs['job_location'].astype(str).str.lower().str.strip()
    candidates['candidate_location'] = candidates['candidate_location'].astype(str).str.lower().str.strip()

    jobs['job_posted_date'] = pd.to_datetime(jobs['job_posted_date'], errors='coerce')
    df_train['application_date'] = pd.to_datetime(df_train['application_date'], errors='coerce')
    df_test['application_date'] = pd.to_datetime(df_test['application_date'], errors='coerce')

    jobs['min_years_experience'] = jobs['min_years_experience'].fillna(0)
    jobs['max_years_experience'] = jobs['max_years_experience'].fillna(50)
    jobs['remote_allowed'] = jobs['remote_allowed'].fillna('No')
    
    candidates['years_experience'] = candidates['years_experience'].fillna(0)
    candidates['willing_to_relocate'] = candidates['willing_to_relocate'].fillna('No')
    candidates['english_score'] = candidates['english_score'].fillna(candidates['english_score'].median())
    candidates['edu_score'] = candidates['edu_score'].fillna(candidates['edu_score'].median())
    candidates['expected_salary'] = candidates['expected_salary'].fillna(candidates['expected_salary'].median())

    train_merged = df_train.merge(jobs, on='job_id', how='left').merge(candidates, on='candidate_id', how='left')
    test_merged = df_test.merge(jobs, on='job_id', how='left').merge(candidates, on='candidate_id', how='left')
    
    return train_merged, test_merged

train_data, test_data = run_phase_one_two('applications_train.csv', 'applications_test.csv', 'jobs.csv', 'candidates.csv')

print("=== Phase 1 & 2 Sanity Check ===")
print(f"Train Data Shape: {train_data.shape}")
print(f"Test Data Shape: {test_data.shape}")
print("\n[Sample Data Check - First 3 Rows]")
print(train_data[['application_id', 'salary_min_usd', 'english_score', 'edu_score', 'years_experience', 'relevance_label']].head(3))
print("\n[Missing Values in Key Columns]")
print(train_data[['job_id', 'candidate_id', 'salary_min_usd', 'english_score', 'edu_score']].isnull().sum())
print("================================")

=== Phase 1 & 2 Sanity Check ===
Train Data Shape: (118772, 34)
Test Data Shape: (52700, 33)

[Sample Data Check - First 3 Rows]
   application_id  salary_min_usd  english_score  edu_score  years_experience  \
0             206          5040.0            4.0          3               8.0   
1             207          5040.0            5.0          4               5.0   
2             208          5040.0            3.0          2               2.0   

   relevance_label  
0                2  
1                1  
2                2  

[Missing Values in Key Columns]
job_id            0
candidate_id      0
salary_min_usd    0
english_score     0
edu_score         0
dtype: int64


In [9]:
def run_phase_three(train_df, test_df):
    def engineer_features(df):
        # 1. Skill Match Ratio
        def get_skill_match(row):
            if pd.isna(row['required_skills']) or pd.isna(row['skills']):
                return 0.0
            req = set(row['required_skills'].split('|'))
            cand = set(row['skills'].split('|'))
            if len(req) == 0: 
                return 0.0
            return len(req.intersection(cand)) / len(req)
        
        df['skill_match_ratio'] = df.apply(get_skill_match, axis=1)

        # 2. Salary Gap
        df['job_avg_salary_usd'] = df[['salary_min_usd', 'salary_max_usd']].mean(axis=1)
        df['salary_gap'] = df['expected_salary'] - df['job_avg_salary_usd']
        df['salary_gap'] = df['salary_gap'].fillna(0) # پر کردن گپ‌های احتمالی

        # 3. Experience Gap
        df['experience_gap'] = df['years_experience'] - df['min_years_experience']

        # 4. Location Match
        df['location_match'] = ((df['candidate_location'] == df['job_location']) | 
                                (df['remote_allowed'].str.lower() == 'yes') | 
                                (df['willing_to_relocate'].str.lower() == 'yes')).astype(int)

        # 5. Application Speed (Days to Apply)
        df['days_to_apply'] = (df['application_date'] - df['job_posted_date']).dt.days
        df['days_to_apply'] = df['days_to_apply'].fillna(df['days_to_apply'].median())
        df.loc[df['days_to_apply'] < 0, 'days_to_apply'] = 0 # اصلاح باگ‌های زمانی منفی
        
        return df
    
    print("Engineering features for train data...")
    train_eng = engineer_features(train_df.copy())
    print("Engineering features for test data...")
    test_eng = engineer_features(test_df.copy())
    
    return train_eng, test_eng

# اجرای فاز سوم
train_data_eng, test_data_eng = run_phase_three(train_data, test_data)

# --- Sanity Check فاز 3 ---
print("\n=== Phase 3 Sanity Check ===")
features_to_check = ['skill_match_ratio', 'salary_gap', 'experience_gap', 'location_match', 'days_to_apply']
print("[Sample Data Check - New Features]")
print(train_data_eng[features_to_check + ['relevance_label']].head())
print("\n[Missing Values in New Features]")
print(train_data_eng[features_to_check].isnull().sum())
print("============================")

Engineering features for train data...
Engineering features for test data...

=== Phase 3 Sanity Check ===
[Sample Data Check - New Features]
   skill_match_ratio     salary_gap  experience_gap  location_match  \
0                0.2  110505.275330             4.0               0   
1                0.0   80816.039783             1.0               0   
2                0.6   85805.832040            -2.0               0   
3                0.0   53624.450025            -2.0               1   
4                0.0   45015.932462            -3.0               1   

   days_to_apply  relevance_label  
0           24.0                2  
1           14.0                1  
2           24.0                2  
3           24.0                1  
4           14.0                0  

[Missing Values in New Features]
skill_match_ratio    0
salary_gap           0
experience_gap       0
location_match       0
days_to_apply        0
dtype: int64


In [10]:
!pip install lightgbm scikit-learn

/media/E/term6/DS/Datasets/task 3 - datasets/.venv/bin/pip: 2: exec: /home/sajjad/Downloads/Datasets/task 3 - datasets/.venv/bin/python: not found


In [11]:
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
import numpy as np
import pandas as pd

def run_phase_four(train_df):
    # ۱. حذف ستون‌های متنی، تاریخ‌ها و IDها (مدل فقط عدد می‌فهمد)
    cols_to_drop = [
        'application_id', 'candidate_id', 'application_date', 'job_posted_date',
        'job_title', 'required_skills', 'salary_currency', 'job_location', 'remote_allowed', 
        'company_size', 'industry', 'current_title', 'skills', 'education_level', 
        'university', 'previous_companies', 'certifications', 'english_proficiency', 
        'candidate_location', 'willing_to_relocate', 'account_created_date',
        'relevance_label' # لیبل هدف را نباید به عنوان فیچر به مدل بدهیم!
    ]
    
    # استخراج فیچرهای نهایی برای آموزش
    features = [c for c in train_df.columns if c not in cols_to_drop]
    
    X = train_df[features]
    y = train_df['relevance_label']
    groups = train_df['job_id']
    
    # ۲. تعریف GroupKFold برای ایزوله کردن شغل‌ها
    gkf = GroupKFold(n_splits=5)
    oof_predictions = np.zeros(len(train_df))
    
    # ۳. تعریف مدل رگرسیون LightGBM
    model = lgb.LGBMRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=7,
        random_state=42,
        n_jobs=-1
    )
    
    # ذخیره اهمیت ویژگی‌ها برای تحلیل
    feature_importances = np.zeros(len(features))
    
    print(f"Training on {len(features)} features...")
    
    # ۴. حلقه آموزش و اعتبارسنجی
    for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
        
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
        )
        
        # پیش‌بینی روی دیتای ولیدیشن
        oof_predictions[val_idx] = model.predict(X_val)
        feature_importances += model.feature_importances_ / gkf.n_splits
        print(f"Fold {fold+1} completed.")
        
    train_df['predicted_score'] = oof_predictions
    
    # ساخت دیتافریم اهمیت ویژگی‌ها
    fi_df = pd.DataFrame({
        'Feature': features,
        'Importance': feature_importances
    }).sort_values(by='Importance', ascending=False)
    
    return train_df, fi_df, model, features

# اجرای فاز ۴
train_data_scored, feature_importance_df, final_model, final_features = run_phase_four(train_data_eng)

# --- Sanity Check فاز 4 ---
print("\n=== Phase 4 Sanity Check ===")
print("[Top 10 Most Important Features]")
print(feature_importance_df.head(10))
print("\n[Sample Predictions vs Actual Labels]")
print(train_data_scored[['job_id', 'candidate_id', 'relevance_label', 'predicted_score']].head())
print("============================")

Training on 18 features...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.024169 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2241
[LightGBM] [Info] Number of data points in the train set: 95017, number of used features: 18
[LightGBM] [Info] Start training from score 1.148952
Fold 1 completed.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.025389 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2242
[LightGBM] [Info] Number of data points in the train set: 95017, number of used features: 18
[LightGBM] [Info] Start training from score 1.148752
Fold 2 completed.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.023123 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2241
[LightGBM] [Info] Number of data points in the train set:

In [12]:

def calculate_metrics(df):
    ndcg_scores = []
    ap_scores = []
    
    for job_id, group in df.groupby('job_id'):
        y_true = group['relevance_label'].values
        y_pred = group['predicted_score'].values
        
        order = np.argsort(y_pred)[::-1]
        y_true_sorted = y_true[order]
        
        ideal_order = np.argsort(y_true)[::-1]
        y_ideal_sorted = y_true[ideal_order]
        
        def dcg_at_k(y, k):
            y_k = y[:k]
            gains = 2**y_k - 1
            discounts = np.log2(np.arange(len(y_k)) + 2)
            return np.sum(gains / discounts)
        
        dcg = dcg_at_k(y_true_sorted, 10)
        idcg = dcg_at_k(y_ideal_sorted, 10)
        
        if idcg == 0:
            ndcg_scores.append(0.0)
        else:
            ndcg_scores.append(dcg / idcg)
        
        y_true_binary = (y_true >= 3).astype(int)
        y_true_sorted_binary = (y_true_sorted >= 3).astype(int)
        
        total_relevant = np.sum(y_true_binary)
        
        if total_relevant == 0:
            ap_scores.append(0.0)
            continue
            
        y_k_bin = y_true_sorted_binary[:5]
        precisions = []
        rel_count = 0
        
        for i, rel in enumerate(y_k_bin):
            if rel == 1:
                rel_count += 1
                precisions.append(rel_count / (i + 1))
        
        if len(precisions) == 0:
            ap_scores.append(0.0)
        else:
            ap = np.sum(precisions) / min(total_relevant, 5)
            ap_scores.append(ap)
            
    return np.mean(ndcg_scores), np.mean(ap_scores)

mean_ndcg, mean_ap = calculate_metrics(train_data_scored)
print(f"Mean NDCG@10: {mean_ndcg:.4f}")
print(f"MAP@5: {mean_ap:.4f}")

Mean NDCG@10: 0.7731
MAP@5: 0.5504


In [13]:

def run_phase_six(test_df, model, features):
    print("Preparing test data for prediction...")
    
    # جدا کردن دقیقاً همان فیچرهایی که مدل با آن‌ها آموزش دیده است
    X_test = test_df[features]
    
    print("Generating predictions...")
    # پیش‌بینی امتیازها برای دیتای تست
    test_predictions = model.predict(X_test)
    
    # ساخت دیتافریم نهایی طبق استاندارد کوئرا
    submission_df = pd.DataFrame({
        'application_id': test_df['application_id'],
        'score': test_predictions
    })
    
    # ذخیره فایل CSV بدون ستون ایندکس
    submission_file_name = 'submission.csv'
    submission_df.to_csv(submission_file_name, index=False)
    
    return submission_df, submission_file_name

# اجرای فاز ۶
final_submission, file_name = run_phase_six(test_data_eng, final_model, final_features)

# --- Sanity Check فاز 6 ---
print("\n=== Phase 6 Sanity Check ===")
print(f"Submission File Shape: {final_submission.shape} (باید دقیقاً 52700 سطر و 2 ستون باشد)")
print("\n[Sample Submission - First 5 Rows]")
print(final_submission.head())
print("\n[Missing Values Check]")
print(final_submission.isnull().sum())
print(f"\n✅ File '{file_name}' successfully generated in your directory!")
print("============================")

Preparing test data for prediction...
Generating predictions...

=== Phase 6 Sanity Check ===
Submission File Shape: (52700, 2) (باید دقیقاً 52700 سطر و 2 ستون باشد)

[Sample Submission - First 5 Rows]
   application_id     score
0               1  2.504817
1               2  1.960901
2               3  1.028975
3               4  1.319815
4               5  1.406372

[Missing Values Check]
application_id    0
score             0
dtype: int64

✅ File 'submission.csv' successfully generated in your directory!


In [14]:
import pandas as pd
import numpy as np
from difflib import SequenceMatcher
from sklearn.model_selection import GroupKFold

def run_phase_seven(train_df, test_df):
    print("1. Forging Weapon 1: Title Similarity...")
    
    def string_similarity(a, b):
        if pd.isna(a) or pd.isna(b):
            return 0.0
        return SequenceMatcher(None, str(a).lower(), str(b).lower()).ratio()

    train_df['title_similarity'] = train_df.apply(lambda x: string_similarity(x['job_title'], x['current_title']), axis=1)
    test_df['title_similarity'] = test_df.apply(lambda x: string_similarity(x['job_title'], x['current_title']), axis=1)

    print("2. Forging Weapon 2: OOF Target Encoding for Industry...")
    gkf = GroupKFold(n_splits=5)
    train_df['industry_te'] = np.nan
    global_mean = train_df['relevance_label'].mean()

    # اجرای Out-of-Fold برای جلوگیری از نشت داده
    for train_idx, val_idx in gkf.split(train_df, train_df['relevance_label'], train_df['job_id']):
        X_tr = train_df.iloc[train_idx]
        X_va = train_df.iloc[val_idx]
        
        # محاسبه میانگین لیبل برای هر صنعت فقط روی دیتای آموزش همون فولد
        industry_means = X_tr.groupby('industry')['relevance_label'].mean()
        
        # مپ کردن این میانگین روی دیتای ولیدیشن
        train_df.loc[val_idx, 'industry_te'] = X_va['industry'].map(industry_means)
        
    train_df['industry_te'] = train_df['industry_te'].fillna(global_mean)
    
    # برای دیتای تست، از میانگین کل دیتای آموزش استفاده می‌کنیم
    test_industry_means = train_df.groupby('industry')['relevance_label'].mean()
    test_df['industry_te'] = test_df['industry'].map(test_industry_means).fillna(global_mean)

    return train_df, test_df

# اجرای فاز 7
train_data_adv, test_data_adv = run_phase_seven(train_data_eng.copy(), test_data_eng.copy())

# --- Sanity Check فاز 7 ---
print("\n=== Phase 7 Sanity Check ===")
print("[Checking Title Similarity & OOF Encoding]")
print(train_data_adv[['job_id', 'job_title', 'current_title', 'title_similarity', 'industry', 'industry_te', 'relevance_label']].head())
print("\n[Missing Values Check]")
print(train_data_adv[['title_similarity', 'industry_te']].isnull().sum())
print("============================")

1. Forging Weapon 1: Title Similarity...
2. Forging Weapon 2: OOF Target Encoding for Industry...

=== Phase 7 Sanity Check ===
[Checking Title Similarity & OOF Encoding]
   job_id      job_title           current_title  title_similarity  \
0       6  Web Developer  Senior DevOps Engineer          0.514286   
1       6  Web Developer           Web Developer          1.000000   
2       6  Web Developer       Backend Developer          0.733333   
3       6  Web Developer      Frontend Developer          0.709677   
4       6  Web Developer            Head of Data          0.240000   

         industry  industry_te  relevance_label  
0  Transportation     1.138577                2  
1  Transportation     1.138577                1  
2  Transportation     1.138577                2  
3  Transportation     1.138577                1  
4  Transportation     1.138577                0  

[Missing Values Check]
title_similarity    0
industry_te         0
dtype: int64


In [15]:

import lightgbm as lgb
from sklearn.model_selection import GroupKFold

def calculate_metrics(df):
    ndcg_scores = []
    ap_scores = []
    
    for job_id, group in df.groupby('job_id'):
        y_true = group['relevance_label'].values
        y_pred = group['predicted_score'].values
        
        order = np.argsort(y_pred)[::-1]
        y_true_sorted = y_true[order]
        
        ideal_order = np.argsort(y_true)[::-1]
        y_ideal_sorted = y_true[ideal_order]
        
        def dcg_at_k(y, k):
            y_k = y[:k]
            gains = 2**y_k - 1
            discounts = np.log2(np.arange(len(y_k)) + 2)
            return np.sum(gains / discounts)
        
        dcg = dcg_at_k(y_true_sorted, 10)
        idcg = dcg_at_k(y_ideal_sorted, 10)
        
        if idcg == 0:
            ndcg_scores.append(0.0)
        else:
            ndcg_scores.append(dcg / idcg)
        
        y_true_binary = (y_true >= 3).astype(int)
        y_true_sorted_binary = (y_true_sorted >= 3).astype(int)
        
        total_relevant = np.sum(y_true_binary)
        
        if total_relevant == 0:
            ap_scores.append(0.0)
            continue
            
        y_k_bin = y_true_sorted_binary[:5]
        precisions = []
        rel_count = 0
        
        for i, rel in enumerate(y_k_bin):
            if rel == 1:
                rel_count += 1
                precisions.append(rel_count / (i + 1))
        
        if len(precisions) == 0:
            ap_scores.append(0.0)
        else:
            ap = np.sum(precisions) / min(total_relevant, 5)
            ap_scores.append(ap)
            
    return np.mean(ndcg_scores), np.mean(ap_scores)

def run_phase_eight(train_df):
    print("Deploying LGBMRanker Engine...")
    
    train_df = train_df.sort_values('job_id').reset_index(drop=True)

    cols_to_drop = [
        'application_id', 'candidate_id', 'application_date', 'job_posted_date',
        'job_title', 'required_skills', 'salary_currency', 'job_location', 'remote_allowed', 
        'company_size', 'industry', 'current_title', 'skills', 'education_level', 
        'university', 'previous_companies', 'certifications', 'english_proficiency', 
        'candidate_location', 'willing_to_relocate', 'account_created_date',
        'relevance_label', 'predicted_score'
    ]
    
    features = [c for c in train_df.columns if c not in cols_to_drop]
    
    X = train_df[features]
    y = train_df['relevance_label']
    groups = train_df['job_id']
    
    gkf = GroupKFold(n_splits=5)
    oof_predictions = np.zeros(len(train_df))
    
    model = lgb.LGBMRanker(
        n_estimators=400,
        learning_rate=0.03,
        max_depth=7,
        random_state=42,
        n_jobs=-1
    )
    
    print(f"Training on {len(features)} advanced features...")
    
    for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        X_tr, y_tr, g_tr = X.iloc[train_idx], y.iloc[train_idx], groups.iloc[train_idx]
        X_va, y_va, g_va = X.iloc[val_idx], y.iloc[val_idx], groups.iloc[val_idx]
        
        sort_tr = np.argsort(g_tr.values)
        X_tr = X_tr.iloc[sort_tr]
        y_tr = y_tr.iloc[sort_tr]
        g_tr = g_tr.iloc[sort_tr]
        
        sort_va = np.argsort(g_va.values)
        X_va = X_va.iloc[sort_va]
        y_va = y_va.iloc[sort_va]
        g_va = g_va.iloc[sort_va]
        
        group_train = g_tr.groupby(g_tr, sort=False).size().values
        group_val = g_va.groupby(g_va, sort=False).size().values
        
        model.fit(
            X_tr, y_tr,
            group=group_train,
            eval_set=[(X_va, y_va)],
            eval_group=[group_val],
            callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
        )
        
        oof_predictions[val_idx] = model.predict(X.iloc[val_idx])
        print(f"Ranker Fold {fold+1} completed.")
        
    train_df['predicted_score'] = oof_predictions
    
    return train_df, model, features

train_data_ranked, final_ranker_model, ranker_features = run_phase_eight(train_data_adv)

mean_ndcg, mean_ap = calculate_metrics(train_data_ranked)
print("\n=== RANKER METRICS CHECK ===")
print(f"New Mean NDCG@10: {mean_ndcg:.4f} (قبلی: 0.7731)")
print(f"New MAP@5: {mean_ap:.4f} (قبلی: 0.5504)")
print("============================")

Deploying LGBMRanker Engine...
Training on 20 advanced features...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000886 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2585
[LightGBM] [Info] Number of data points in the train set: 95017, number of used features: 20
Ranker Fold 1 completed.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000876 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2586
[LightGBM] [Info] Number of data points in the train set: 95017, number of used features: 20
Ranker Fold 2 completed.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001058 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory i

In [16]:

def apply_group_rank_features(train_df, test_df):
    cols_to_rank = [
        'experience_gap', 'salary_gap', 'skill_match_ratio', 
        'title_similarity', 'days_to_apply', 'english_score', 'edu_score'
    ]
    
    def add_rank_features(df):
        g = df.groupby('job_id')
        for c in cols_to_rank:
            if c in df.columns:
                df[f"{c}_grank"] = g[c].rank(pct=True)
                df[f"{c}_gzscore"] = g[c].transform(lambda x: (x - x.mean()) / (x.std() + 1e-6))
        return df
        
    train_out = add_rank_features(train_df.copy())
    test_out = add_rank_features(test_df.copy())
    
    return train_out, test_out

train_data_p9, test_data_p9 = apply_group_rank_features(train_data_adv, test_data_adv)

print("\n=== Phase 9 Sanity Check ===")
print("[Sample Data Check - Rank & Z-Score Features]")
check_cols = ['job_id', 'experience_gap', 'experience_gap_grank', 'experience_gap_gzscore', 'relevance_label']
print(train_data_p9[check_cols].head())
print("============================")


=== Phase 9 Sanity Check ===
[Sample Data Check - Rank & Z-Score Features]
   job_id  experience_gap  experience_gap_grank  experience_gap_gzscore  \
0       6             4.0               0.81250                0.548345   
1       6             1.0               0.59375               -0.078335   
2       6            -2.0               0.31250               -0.705014   
3       6            -2.0               0.31250               -0.705014   
4       6            -3.0               0.18750               -0.913908   

   relevance_label  
0                2  
1                1  
2                2  
3                1  
4                0  


In [17]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold

def calculate_metrics(df):
    ndcg_scores = []
    ap_scores = []
    
    for job_id, group in df.groupby('job_id'):
        y_true = group['relevance_label'].values
        y_pred = group['predicted_score'].values
        order = np.argsort(y_pred)[::-1]
        y_true_sorted = y_true[order]
        ideal_order = np.argsort(y_true)[::-1]
        y_ideal_sorted = y_true[ideal_order]
        
        def dcg_at_k(y, k):
            y_k = y[:k]
            gains = 2**y_k - 1
            discounts = np.log2(np.arange(len(y_k)) + 2)
            return np.sum(gains / discounts)
        
        dcg = dcg_at_k(y_true_sorted, 10)
        idcg = dcg_at_k(y_ideal_sorted, 10)
        ndcg_scores.append(0.0 if idcg == 0 else dcg / idcg)
        
        y_true_binary = (y_true >= 3).astype(int)
        y_true_sorted_binary = (y_true_sorted >= 3).astype(int)
        total_relevant = np.sum(y_true_binary)
        
        if total_relevant == 0:
            ap_scores.append(0.0)
            continue
            
        y_k_bin = y_true_sorted_binary[:5]
        precisions = []
        rel_count = 0
        for i, rel in enumerate(y_k_bin):
            if rel == 1:
                rel_count += 1
                precisions.append(rel_count / (i + 1))
        ap_scores.append(0.0 if len(precisions) == 0 else np.sum(precisions) / min(total_relevant, 5))
            
    return np.mean(ndcg_scores), np.mean(ap_scores)

def run_phase_ten(train_df):
    print("Firing up Ranker with Group Rank Features...")
    train_df = train_df.sort_values('job_id').reset_index(drop=True)
    
    cols_to_drop = [
        'application_id', 'candidate_id', 'application_date', 'job_posted_date',
        'job_title', 'required_skills', 'salary_currency', 'job_location', 'remote_allowed', 
        'company_size', 'industry', 'current_title', 'skills', 'education_level', 
        'university', 'previous_companies', 'certifications', 'english_proficiency', 
        'candidate_location', 'willing_to_relocate', 'account_created_date',
        'relevance_label', 'predicted_score'
    ]
    
    features = [c for c in train_df.columns if c not in cols_to_drop]
    X = train_df[features]
    y = train_df['relevance_label']
    groups = train_df['job_id']
    
    gkf = GroupKFold(n_splits=5)
    oof_predictions = np.zeros(len(train_df))
    
    model = lgb.LGBMRanker(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=7,
        random_state=42,
        n_jobs=-1
    )
    
    for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        X_tr, y_tr, g_tr = X.iloc[train_idx], y.iloc[train_idx], groups.iloc[train_idx]
        X_va, y_va, g_va = X.iloc[val_idx], y.iloc[val_idx], groups.iloc[val_idx]
        
        sort_tr = np.argsort(g_tr.values)
        X_tr, y_tr, g_tr = X_tr.iloc[sort_tr], y_tr.iloc[sort_tr], g_tr.iloc[sort_tr]
        
        sort_va = np.argsort(g_va.values)
        X_va, y_va, g_va = X_va.iloc[sort_va], y_va.iloc[sort_va], g_va.iloc[sort_va]
        
        group_train = g_tr.groupby(g_tr, sort=False).size().values
        group_val = g_va.groupby(g_va, sort=False).size().values
        
        model.fit(
            X_tr, y_tr, group=group_train,
            eval_set=[(X_va, y_va)], eval_group=[group_val],
            callbacks=[lgb.early_stopping(stopping_rounds=40, verbose=False)]
        )
        
        oof_predictions[val_idx] = model.predict(X.iloc[val_idx])
        print(f"Ranker Fold {fold+1} completed.")
        
    train_df['predicted_score'] = oof_predictions
    return train_df, model, features

train_data_final, final_ranker_model, final_features = run_phase_ten(train_data_p9)

mean_ndcg, mean_ap = calculate_metrics(train_data_final)
print("\n=== RANKER PHASE 10 METRICS ===")
print(f"Final Mean NDCG@10: {mean_ndcg:.5f} (قبلی: 0.7763)")
print(f"Final MAP@5: {mean_ap:.5f} (قبلی: 0.5564)")
print("===============================")

Firing up Ranker with Group Rank Features...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002554 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6153
[LightGBM] [Info] Number of data points in the train set: 95017, number of used features: 34
Ranker Fold 1 completed.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.029635 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6155
[LightGBM] [Info] Number of data points in the train set: 95017, number of used features: 34
Ranker Fold 2 completed.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.012137 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6150
[LightGBM] [Info] Number of data points in the train set: 95018, nu

In [18]:
import numpy as np
import pandas as pd
from collections import Counter

def tokenize_pipe(s):
    if pd.isna(s):
        return set()
    return {t.strip().lower() for t in str(s).split("|") if t.strip()}

def tokenize_title(s):
    if pd.isna(s):
        return set()
    return {w for w in str(s).lower().replace(".", " ").split() if len(w) > 1}

def run_phase_eleven(train_df, test_df):
    def build_new_features(df):
        cand_sk = df["skills"].apply(tokenize_pipe)
        req_sk = df["required_skills"].apply(tokenize_pipe)
        certs = df["certifications"].apply(tokenize_pipe)
        job_tok = df["job_title"].apply(tokenize_title)
        cur_tok = df["current_title"].apply(tokenize_title)

        df["skill_jaccard"] = [len(a & b) / len(a | b) if (a | b) else 0.0 for a, b in zip(cand_sk, req_sk)]
        df["missing_required_skills"] = [len(b - a) for a, b in zip(cand_sk, req_sk)]
        df["extra_skills"] = [len(a - b) for a, b in zip(cand_sk, req_sk)]
        df["cert_skill_overlap"] = [len(c & b) for c, b in zip(certs, req_sk)]
        df["cert_title_overlap"] = [len(c & j) for c, j in zip(certs, job_tok)]
        df["n_certifications"] = certs.apply(len)
        df["title_token_jaccard"] = [len(a & b) / len(a | b) if (a | b) else 0.0 for a, b in zip(job_tok, cur_tok)]

        dfreq = Counter()
        for s in req_sk:
            dfreq.update(s)
        n_docs = len(req_sk)
        idf = {k: np.log((1 + n_docs) / (1 + v)) + 1 for k, v in dfreq.items()}

        vals = []
        for a, b in zip(cand_sk, req_sk):
            if not b:
                vals.append(0.0)
                continue
            num = sum(idf.get(t, 0.0) for t in (a & b))
            den = sum(idf.get(t, 0.0) for t in b)
            vals.append(num / den if den else 0.0)
        df["skill_match_idf"] = vals

        exp = df["years_experience"]
        lo = df["min_years_experience"]
        hi = df["max_years_experience"]
        df["in_experience_band"] = ((exp >= lo) & (exp <= hi)).astype(int)
        df["overqualified"] = (exp > hi).astype(int)
        df["underqualified"] = (exp < lo).astype(int)
        df["exp_band_distance"] = np.where(exp < lo, lo - exp, np.where(exp > hi, exp - hi, 0))

        cand_sal = df["expected_salary"]
        jmin = df["salary_min_usd"]
        jmax = df["salary_max_usd"]
        df["salary_in_band"] = ((cand_sal >= jmin) & (cand_sal <= jmax)).astype(int)
        df["salary_over_budget"] = (cand_sal > jmax).astype(int)
        df["salary_band_distance"] = np.where(cand_sal < jmin, jmin - cand_sal, np.where(cand_sal > jmax, cand_sal - jmax, 0))

        prev = df["previous_companies"].apply(tokenize_pipe)
        freq = Counter()
        for s in prev:
            freq.update(s)
        df["max_company_freq"] = [max((freq[c] for c in s), default=0) for s in prev]
        df["mean_company_freq"] = [np.mean([freq[c] for c in s]) if s else 0.0 for s in prev]
        df["n_previous_companies"] = prev.apply(len)
        df["completeness_x_skill"] = df["profile_completeness"] * df["skill_jaccard"]
        
        return df

    train_out = build_new_features(train_df.copy())
    test_out = build_new_features(test_df.copy())
    
    return train_out, test_out

train_data_p11, test_data_p11 = run_phase_eleven(train_data_p9, test_data_p9)

print("\n=== Phase 11 Sanity Check ===")
print(f"Total Features Now: {train_data_p11.shape[1]}")
check_cols = ['job_id', 'skill_jaccard', 'skill_match_idf', 'overqualified', 'max_company_freq']
print(train_data_p11[check_cols].head())
print("=============================")


=== Phase 11 Sanity Check ===
Total Features Now: 76
   job_id  skill_jaccard  skill_match_idf  overqualified  max_company_freq
0       6       0.071429         0.204468              0              2649
1       6       0.000000         0.000000              0              2691
2       6       0.333333         0.616071              0                 0
3       6       0.000000         0.000000              0                 0
4       6       0.000000         0.000000              0              2772


In [19]:

def calculate_metrics(df):
    ndcg_scores = []
    ap_scores = []
    for job_id, group in df.groupby('job_id'):
        y_true = group['relevance_label'].values
        y_pred = group['predicted_score'].values
        order = np.argsort(y_pred)[::-1]
        y_true_sorted = y_true[order]
        ideal_order = np.argsort(y_true)[::-1]
        y_ideal_sorted = y_true[ideal_order]
        
        def dcg_at_k(y, k):
            y_k = y[:k]
            gains = 2**y_k - 1
            discounts = np.log2(np.arange(len(y_k)) + 2)
            return np.sum(gains / discounts)
        
        dcg = dcg_at_k(y_true_sorted, 10)
        idcg = dcg_at_k(y_ideal_sorted, 10)
        ndcg_scores.append(0.0 if idcg == 0 else dcg / idcg)
        
        y_true_binary = (y_true >= 3).astype(int)
        y_true_sorted_binary = (y_true_sorted >= 3).astype(int)
        total_relevant = np.sum(y_true_binary)
        
        if total_relevant == 0:
            ap_scores.append(0.0)
            continue
            
        y_k_bin = y_true_sorted_binary[:5]
        precisions = []
        rel_count = 0
        for i, rel in enumerate(y_k_bin):
            if rel == 1:
                rel_count += 1
                precisions.append(rel_count / (i + 1))
        ap_scores.append(0.0 if len(precisions) == 0 else np.sum(precisions) / min(total_relevant, 5))
            
    return np.mean(ndcg_scores), np.mean(ap_scores)

train_data_p11 = train_data_p11.sort_values('job_id').reset_index(drop=True)
test_data_p11 = test_data_p11.sort_values('job_id').reset_index(drop=True)

cols_to_drop = [
    'application_id', 'candidate_id', 'application_date', 'job_posted_date',
    'job_title', 'required_skills', 'salary_currency', 'job_location', 'remote_allowed', 
    'company_size', 'industry', 'current_title', 'skills', 'education_level', 
    'university', 'previous_companies', 'certifications', 'english_proficiency', 
    'candidate_location', 'willing_to_relocate', 'account_created_date',
    'relevance_label', 'predicted_score'
]

features = [c for c in train_data_p11.columns if c not in cols_to_drop]

X = train_data_p11[features]
y = train_data_p11['relevance_label']
groups = train_data_p11['job_id']

gkf = GroupKFold(n_splits=5)
oof_predictions = np.zeros(len(train_data_p11))

lgb_params = {
    'n_estimators': 1500,
    'learning_rate': 0.03,
    'num_leaves': 63,
    'max_depth': 7,
    'min_child_samples': 30,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.5,
    'lambdarank_truncation_level': 12,
    'random_state': 42,
    'n_jobs': -1
}

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    X_tr, y_tr, g_tr = X.iloc[train_idx], y.iloc[train_idx], groups.iloc[train_idx]
    X_va, y_va, g_va = X.iloc[val_idx], y.iloc[val_idx], groups.iloc[val_idx]
    
    sort_tr = np.argsort(g_tr.values)
    X_tr, y_tr, g_tr = X_tr.iloc[sort_tr], y_tr.iloc[sort_tr], g_tr.iloc[sort_tr]
    
    sort_va = np.argsort(g_va.values)
    X_va, y_va, g_va = X_va.iloc[sort_va], y_va.iloc[sort_va], g_va.iloc[sort_va]
    
    group_train = g_tr.groupby(g_tr, sort=False).size().values
    group_val = g_va.groupby(g_va, sort=False).size().values
    
    model = lgb.LGBMRanker(**lgb_params)
    
    model.fit(
        X_tr, y_tr, group=group_train,
        eval_set=[(X_va, y_va)], eval_group=[group_val],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    oof_predictions[val_idx] = model.predict(X.iloc[val_idx])
    print(f"Ultimate Ranker Fold {fold+1} completed.")

train_data_p11['predicted_score'] = oof_predictions

mean_ndcg, mean_ap = calculate_metrics(train_data_p11)
print("\n=== ULTIMATE MODEL METRICS ===")
print(f"Ultimate NDCG@10: {mean_ndcg:.5f}")
print(f"Ultimate MAP@5: {mean_ap:.5f}")
print("==============================")

final_model_full = lgb.LGBMRanker(**lgb_params)
sort_full = np.argsort(groups.values)
X_full, y_full, groups_full = X.iloc[sort_full], y.iloc[sort_full], groups.iloc[sort_full]
group_train_full = groups_full.groupby(groups_full, sort=False).size().values

final_model_full.fit(X_full, y_full, group=group_train_full)

X_test = test_data_p11[features]
test_predictions_ultimate = final_model_full.predict(X_test)

submission_ultimate_df = pd.DataFrame({
    'application_id': test_data_p11['application_id'],
    'score': test_predictions_ultimate
})

submission_ultimate_df.to_csv('submission_ultimate.csv', index=False)
print("\n✅ File 'submission_ultimate.csv' is ready for Quera!")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009153 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7393
[LightGBM] [Info] Number of data points in the train set: 95017, number of used features: 51
Ultimate Ranker Fold 1 completed.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.032589 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7395
[LightGBM] [Info] Number of data points in the train set: 95017, number of used features: 51
Ultimate Ranker Fold 2 completed.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.028679 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7391
[LightGBM] [Info] Number of data points in the train set: 95018, number of used features: 51
U

In [20]:

def run_phase_thirteen(train_df, test_df):
    def add_explicit_interactions(df):
        if "in_experience_band" in df.columns:
            df["idf_x_inband"] = df["skill_match_idf"] * df["in_experience_band"]
        df["idf_x_salband"] = df["skill_match_idf"] * df["salary_band_distance"]
        df["expgrank_x_salgrank"] = df["experience_gap_grank"] * df["salary_gap_grank"]
        df["title_x_expgrank"] = df["title_token_jaccard"] * df["experience_gap_grank"]
        df["match_per_salovershoot"] = df["skill_match_idf"] / (df["salary_band_distance"] + 1.0)
        df["match_per_expgap"] = df["skill_match_idf"] / (df["exp_band_distance"] + 1.0)
        return df

    train_out = add_explicit_interactions(train_df.copy())
    test_out = add_explicit_interactions(test_df.copy())

    new_cols = [
        "idf_x_inband", "idf_x_salband", "expgrank_x_salgrank",
        "title_x_expgrank", "match_per_salovershoot", "match_per_expgap"
    ]

    def add_group_rank_for_new(df, cols):
        g = df.groupby("job_id")
        for c in cols:
            if c in df.columns:
                df[f"{c}_grank"] = g[c].rank(pct=True)
                df[f"{c}_gzscore"] = g[c].transform(lambda x: (x - x.mean()) / (x.std() + 1e-6))
        return df

    train_out = add_group_rank_for_new(train_out, new_cols)
    test_out = add_group_rank_for_new(test_out, new_cols)

    return train_out, test_out

train_data_p13, test_data_p13 = run_phase_thirteen(train_data_p11, test_data_p11)

print("\n=== Phase 13 Sanity Check ===")
print(f"Total Features Now: {train_data_p13.shape[1]}")
check_cols = ['job_id', 'idf_x_inband', 'expgrank_x_salgrank', 'match_per_salovershoot_grank']
print(train_data_p13[check_cols].head())
print("=============================")


=== Phase 13 Sanity Check ===
Total Features Now: 94
   job_id  idf_x_inband  expgrank_x_salgrank  match_per_salovershoot_grank
0       6      0.204468             0.710938                        0.6250
1       6      0.000000             0.259766                        0.3125
2       6      0.000000             0.175781                        1.0000
3       6      0.000000             0.058594                        0.3125
4       6      0.000000             0.011719                        0.3125


In [21]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

def run_phase_fourteen(train_df, test_df):
    def oof_te_highcard(train, test, cat_col, group_col="job_id", label="relevance_label", n_splits=5, k=20, noise=0.01, min_count=1, seed=42):
        tr = train.copy()
        tr["_nl"] = tr.groupby(group_col)[label].transform(lambda x: x.rank(pct=True) if len(x) > 1 else 0.5)
        gmean = tr["_nl"].mean()
        rng = np.random.default_rng(seed)
        oof = np.full(len(tr), gmean, dtype=float)
        gkf = GroupKFold(n_splits=n_splits)

        for tr_idx, val_idx in gkf.split(tr, groups=tr[group_col]):
            fold = tr.iloc[tr_idx]
            agg = fold.groupby(cat_col)["_nl"].agg(["mean", "count"])
            agg = agg[agg["count"] >= min_count]
            smooth = (agg["mean"] * agg["count"] + gmean * k) / (agg["count"] + k)
            mapped = tr.iloc[val_idx][cat_col].map(smooth).fillna(gmean).values
            mapped = mapped * (1 + rng.normal(0, noise, size=len(mapped)))
            oof[val_idx] = mapped

        full = tr.groupby(cat_col)["_nl"].agg(["mean", "count"])
        full = full[full["count"] >= min_count]
        full_s = (full["mean"] * full["count"] + gmean * k) / (full["count"] + k)
        test_enc = test[cat_col].map(full_s).fillna(gmean).values
        
        train_freq = train[cat_col].map(train[cat_col].value_counts()).fillna(0)
        test_freq = test[cat_col].map(train[cat_col].value_counts()).fillna(0)
        
        return oof, test_enc, train_freq, test_freq

    high_card_cols = ['current_title', 'job_location', 'university']
    
    for col in high_card_cols:
        tr_enc, te_enc, tr_freq, te_freq = oof_te_highcard(train_df, test_df, col)
        train_df[f"{col}_te"] = tr_enc
        test_df[f"{col}_te"] = te_enc
        train_df[f"{col}_freq"] = tr_freq
        test_df[f"{col}_freq"] = te_freq

    return train_df, test_df

train_data_p14, test_data_p14 = run_phase_fourteen(train_data_p13.copy(), test_data_p13.copy())

print("\n=== Phase 14 Sanity Check ===")
print(f"Total Features Now: {train_data_p14.shape[1]}")
check_cols = ['job_id', 'current_title', 'current_title_te', 'current_title_freq', 'university_te']
print(train_data_p14[check_cols].head())
print("=============================")


=== Phase 14 Sanity Check ===
Total Features Now: 100
   job_id           current_title  current_title_te  current_title_freq  \
0       6  Senior DevOps Engineer          0.513909                2919   
1       6           Web Developer          0.559025                2898   
2       6       Backend Developer          0.533566                2904   
3       6      Frontend Developer          0.537263                3037   
4       6            Head of Data          0.493468                2928   

   university_te  
0       0.516820  
1       0.529863  
2       0.513406  
3       0.522216  
4       0.519232  


In [22]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold

def run_phase_fifteen(train_df):
    print("Phase 15: Pruning Features via Null Importance (Fast Mode)...")
    cols_to_drop = [
        'application_id', 'candidate_id', 'application_date', 'job_posted_date',
        'job_title', 'required_skills', 'salary_currency', 'job_location', 'remote_allowed', 
        'company_size', 'industry', 'current_title', 'skills', 'education_level', 
        'university', 'previous_companies', 'certifications', 'english_proficiency', 
        'candidate_location', 'willing_to_relocate', 'account_created_date',
        'relevance_label', 'predicted_score'
    ]
    
    features = [c for c in train_df.columns if c not in cols_to_drop]
    X = train_df[features]
    y = train_df['relevance_label'].values
    groups = train_df['job_id'].values

    # 1. Train with Actual Targets
    print("Evaluating Actual Importances...")
    m_actual = lgb.LGBMRanker(n_estimators=200, learning_rate=0.05, random_state=42, n_jobs=-1)
    sort_idx = np.argsort(groups, kind='stable')
    _, counts = np.unique(groups[sort_idx], return_counts=True)
    
    m_actual.fit(X.iloc[sort_idx], y[sort_idx], group=counts)
    actual_imp = m_actual.booster_.feature_importance(importance_type="gain")

    # 2. Train with Null (Shuffled) Targets
    print("Evaluating Null Importances (Noise)...")
    n_runs = 15
    null_imp = np.zeros((n_runs, len(features)))
    rng = np.random.default_rng(42)
    
    for i in range(n_runs):
        y_shuf = y.copy()
        df_shuf = pd.DataFrame({"y": y, "g": groups, "pos": np.arange(len(y))})
        for _, grp in df_shuf.groupby("g"):
            vals = grp["y"].values.copy()
            rng.shuffle(vals)
            y_shuf[grp["pos"].values] = vals
            
        m_null = lgb.LGBMRanker(n_estimators=200, learning_rate=0.05, random_state=42+i, n_jobs=-1)
        m_null.fit(X.iloc[sort_idx], y_shuf[sort_idx], group=counts)
        null_imp[i] = m_null.booster_.feature_importance(importance_type="gain")

    # 3. Keep features that consistently beat noise
    keep_features = []
    dropped_features = []
    
    for j, f in enumerate(features):
        null_mean = null_imp[:, j].mean()
        # Rule: Actual importance must be at least 50% larger than random noise mean
        if actual_imp[j] > null_mean * 1.5:  
            keep_features.append(f)
        else:
            dropped_features.append(f)
            
    print("\n=== Feature Pruning Results ===")
    print(f"Original Features: {len(features)}")
    print(f"Kept Features (Valuable): {len(keep_features)}")
    print(f"Dropped Features (Noise): {len(dropped_features)}")
    print("===============================\n")
    
    return keep_features

# اجرای فاز 15
final_pruned_features = run_phase_fifteen(train_data_p14)

Phase 15: Pruning Features via Null Importance (Fast Mode)...
Evaluating Actual Importances...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007557 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 13017
[LightGBM] [Info] Number of data points in the train set: 118772, number of used features: 75
Evaluating Null Importances (Noise)...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.066490 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 13017
[LightGBM] [Info] Number of data points in the train set: 118772, number of used features: 75
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.027815 seconds.
You can set `force_row_wise=true` to remove the over

In [ ]:
pip install catboost

Note: you may need to restart the kernel to use updated packages.


In [24]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostRanker, Pool
from sklearn.model_selection import GroupKFold
from sklearn.metrics import ndcg_score

def ndcg_by_group(y_true, y_pred, groups, k=10):
    df = pd.DataFrame({"t": y_true, "p": y_pred, "g": groups})
    s = [ndcg_score([gr["t"].values], [gr["p"].values], k=k) for _, gr in df.groupby("g") if len(gr) >= 2]
    return float(np.mean(s))

def group_rank(pred, groups):
    return pd.Series(pred).groupby(groups).rank(pct=True).values

def run_phase_sixteen(train_df, test_df, features):
    print("Phase 16: The Ultimate Ensemble (LightGBM + CatBoost)...")
    
    train_df = train_df.sort_values('job_id').reset_index(drop=True)
    test_df = test_df.sort_values('job_id').reset_index(drop=True)
    
    X = train_df[features]
    y = train_df['relevance_label'].values
    groups = train_df['job_id'].values
    
    lgb_params = {
        'objective': 'lambdarank', 'metric': 'ndcg', 'n_estimators': 1500,
        'learning_rate': 0.03, 'num_leaves': 63, 'max_depth': 7,
        'min_child_samples': 30, 'subsample': 0.8, 'colsample_bytree': 0.8,
        'random_state': 42, 'n_jobs': -1
    }
    
    gkf = GroupKFold(n_splits=5)
    oof_lgb = np.zeros(len(X))
    oof_cat = np.zeros(len(X))
    
    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
        tr_order = np.argsort(groups[tr_idx], kind="stable")
        Xtr, ytr, gtr = X.iloc[tr_idx[tr_order]], y[tr_idx[tr_order]], groups[tr_idx[tr_order]]
        
        va_order = np.argsort(groups[val_idx], kind="stable")
        Xva, yva, gva = X.iloc[val_idx[va_order]], y[val_idx[va_order]], groups[val_idx[va_order]]
        
        _, c_tr = np.unique(gtr, return_counts=True)
        _, c_va = np.unique(gva, return_counts=True)
        
        lm = lgb.LGBMRanker(**lgb_params)
        lm.fit(Xtr, ytr, group=c_tr, eval_set=[(Xva, yva)], eval_group=[c_va], callbacks=[lgb.early_stopping(50, verbose=False)])
        oof_lgb[val_idx] = lm.predict(X.iloc[val_idx])
        
        cm = CatBoostRanker(loss_function="YetiRank", eval_metric="NDCG:top=10", iterations=1000, learning_rate=0.03, depth=7, random_seed=42, verbose=0, early_stopping_rounds=50)
        cm.fit(Pool(Xtr, ytr, group_id=gtr), eval_set=Pool(Xva, yva, group_id=gva))
        oof_cat[val_idx] = cm.predict(X.iloc[val_idx])
        
        print(f"Ensemble Fold {fold+1} completed.")

    r_lgb = group_rank(oof_lgb, groups)
    r_cat = group_rank(oof_cat, groups)

    best_score, best_w = -1, 0.5
    for w in np.linspace(0, 1, 21):
        sc = ndcg_by_group(y, w * r_lgb + (1 - w) * r_cat, groups)
        if sc > best_score:
            best_score, best_w = sc, w

    print(f"\nLGB OOF NDCG: {ndcg_by_group(y, oof_lgb, groups):.5f}")
    print(f"CAT OOF NDCG: {ndcg_by_group(y, oof_cat, groups):.5f}")
    print(f"BLEND NDCG:   {best_score:.5f} (w_lgb={best_w:.2f})")
    
    print("\nTraining Final Models on Full Data...")
    _, c_full = np.unique(groups, return_counts=True)
    
    final_lgb = lgb.LGBMRanker(**lgb_params)
    final_lgb.fit(X, y, group=c_full)
    
    final_cat = CatBoostRanker(loss_function="YetiRank", iterations=1000, learning_rate=0.03, depth=7, random_seed=42, verbose=0)
    final_cat.fit(Pool(X, y, group_id=groups))
    
    X_test = test_df[features]
    test_groups = test_df['job_id'].values
    
    test_pred_lgb = final_lgb.predict(X_test)
    test_pred_cat = final_cat.predict(X_test)
    
    test_r_lgb = group_rank(test_pred_lgb, test_groups)
    test_r_cat = group_rank(test_pred_cat, test_groups)
    
    final_blend = best_w * test_r_lgb + (1 - best_w) * test_r_cat
    
    sub = pd.DataFrame({'application_id': test_df['application_id'], 'score': final_blend})
    sub.to_csv('submission_ensemble_final.csv', index=False)
    print("\n✅ File 'submission_ensemble_final.csv' generated perfectly!")
    return sub

submission_final = run_phase_sixteen(train_data_p14, test_data_p14, final_pruned_features)

Phase 16: The Ultimate Ensemble (LightGBM + CatBoost)...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017228 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5857
[LightGBM] [Info] Number of data points in the train set: 95017, number of used features: 35
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Ensemble Fold 1 completed.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006407 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5860
[LightGBM] [Info] Number of data points in the train set: 95017, number of used features: 35
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Ensemble Fold 2 completed.
[LightGBM] [Info] Auto-choosing r

In [7]:
import numpy as np
import pandas as pd
from collections import Counter
import difflib

import lightgbm as lgb
from catboost import CatBoostRanker, Pool
from sklearn.metrics import ndcg_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import paired_cosine_distances

RNG_SEED = 42

train_app = pd.read_csv('applications_train.csv')
test_app = pd.read_csv('applications_test.csv')
jobs = pd.read_csv('jobs.csv')
candidates = pd.read_csv('candidates.csv')

train_df = train_app.merge(jobs, on='job_id', how='left').merge(candidates, on='candidate_id', how='left')
test_df = test_app.merge(jobs, on='job_id', how='left').merge(candidates, on='candidate_id', how='left')

train_job_ids = set(train_df['job_id'].unique())

eng_map = {'A1': 1, 'A2': 2, 'B1': 3, 'B2': 4, 'C1': 5, 'C2': 6}
edu_map = {'High School': 1, 'Bachelor': 2, 'Master': 3, 'PhD': 4}
exchange_rates = {'USD': 1.0, 'EUR': 1.08, 'IRR': 0.00002}

def _truthy(s):
    return s.fillna(0).astype(str).str.lower().isin(['1', 'true', 'yes', 'y', 't'])

for df in [train_df, test_df]:
    rates = df['salary_currency'].map(exchange_rates).fillna(1.0)
    df['salary_min_usd'] = df['salary_min'] * rates
    df['salary_max_usd'] = df['salary_max'] * rates

_exp_sal_med = train_df['expected_salary'].median()
_sal_min_med = train_df['salary_min_usd'].median()
_sal_max_med = train_df['salary_max_usd'].median()

exp_sal_map = train_df.groupby('current_title')['expected_salary'].median().to_dict()
sal_min_map = train_df.groupby('job_title')['salary_min_usd'].median().to_dict()
sal_max_map = train_df.groupby('job_title')['salary_max_usd'].median().to_dict()
cand_counts = train_df['candidate_id'].value_counts().to_dict()

for i, df in enumerate([train_df, test_df]):
    df['english_score'] = df['english_proficiency'].map(eng_map).fillna(0)
    df['edu_score'] = df['education_level'].map(edu_map).fillna(0)

    df['application_date'] = pd.to_datetime(df['application_date'], dayfirst=True, format='mixed')
    df['job_posted_date'] = pd.to_datetime(df['job_posted_date'], dayfirst=True, format='mixed')
    df['days_to_apply'] = (df['application_date'] - df['job_posted_date']).dt.days.fillna(0)
    
    df['cand_app_count'] = df['candidate_id'].map(cand_counts).fillna(1)

    df['expected_salary'] = df['expected_salary'].fillna(df['current_title'].map(exp_sal_map)).fillna(_exp_sal_med)
    df['salary_min_usd'] = df['salary_min_usd'].fillna(df['job_title'].map(sal_min_map)).fillna(_sal_min_med)
    df['salary_max_usd'] = df['salary_max_usd'].fillna(df['job_title'].map(sal_max_map)).fillna(_sal_max_med)

    df['salary_gap'] = df['expected_salary'] - ((df['salary_min_usd'] + df['salary_max_usd']) / 2)
    df['experience_gap'] = df['years_experience'] - df['min_years_experience']

    df['skill_match_ratio'] = df.apply(
        lambda row: len(set(str(row['skills']).split('|')) & set(str(row['required_skills']).split('|'))) /
                    max(1, len(set(str(row['required_skills']).split('|')))), axis=1
    )
    df['title_similarity'] = df.apply(
        lambda row: difflib.SequenceMatcher(
            None, str(row['job_title']).lower(), str(row['current_title']).lower()).ratio(), axis=1
    )

def tokenize_pipe(s):
    if pd.isna(s):
        return set()
    return {t.strip().lower() for t in str(s).split("|") if t.strip()}

def tokenize_title(s):
    if pd.isna(s):
        return set()
    return {w for w in str(s).lower().replace(".", " ").split() if len(w) > 1}

def build_advanced_features(df, fitted_idf=None, fitted_company_freq=None):
    cand_sk = df["skills"].apply(tokenize_pipe)
    req_sk = df["required_skills"].apply(tokenize_pipe)
    certs = df["certifications"].apply(tokenize_pipe)
    job_tok = df["job_title"].apply(tokenize_title)
    cur_tok = df["current_title"].apply(tokenize_title)

    df["skill_jaccard"] = [len(a & b) / len(a | b) if (a | b) else 0.0 for a, b in zip(cand_sk, req_sk)]
    df["missing_required_skills"] = [len(b - a) for a, b in zip(cand_sk, req_sk)]
    df["extra_skills"] = [len(a - b) for a, b in zip(cand_sk, req_sk)]
    df["cert_skill_overlap"] = [len(c & b) for c, b in zip(certs, req_sk)]
    df["cert_title_overlap"] = [len(c & j) for c, j in zip(certs, job_tok)]
    df["n_certifications"] = certs.apply(len)
    df["title_token_jaccard"] = [len(a & b) / len(a | b) if (a | b) else 0.0 for a, b in zip(job_tok, cur_tok)]

    if fitted_idf is None:
        dfreq = Counter()
        for s in req_sk:
            dfreq.update(s)
        n_docs = len(req_sk)
        idf = {k: np.log((n_docs - v + 0.5) / (v + 0.5) + 1.0) for k, v in dfreq.items()}
    else:
        idf = fitted_idf

    avgdl = np.mean([len(r) for r in req_sk]) if len(req_sk) > 0 else 1.0
    k1, b_param = 1.5, 0.75

    bm25_vals = []
    for cand, req in zip(cand_sk, req_sk):
        if not req:
            bm25_vals.append(0.0)
            continue
        score = 0.0
        doc_len = len(req)
        for t in (cand & req):
            idf_val = idf.get(t, 0.0)
            norm = 1.0 - b_param + b_param * (doc_len / avgdl)
            score += idf_val * (k1 + 1) / (1.0 + k1 * norm)
        bm25_vals.append(score)
    df["skill_match_idf"] = bm25_vals

    exp = df["years_experience"]
    lo = df["min_years_experience"]
    hi = df["max_years_experience"]
    df["in_experience_band"] = ((exp >= lo) & (exp <= hi)).astype(int)
    df["overqualified"] = (exp > hi).astype(int)
    df["underqualified"] = (exp < lo).astype(int)
    df["exp_band_distance"] = np.where(exp < lo, lo - exp, np.where(exp > hi, exp - hi, 0))

    cand_sal = df["expected_salary"]
    jmin = df["salary_min_usd"]
    jmax = df["salary_max_usd"]
    df["salary_in_band"] = ((cand_sal >= jmin) & (cand_sal <= jmax)).astype(int)
    df["salary_over_budget"] = (cand_sal > jmax).astype(int)
    df["salary_band_distance"] = np.where(cand_sal < jmin, jmin - cand_sal,
                                          np.where(cand_sal > jmax, cand_sal - jmax, 0))

    df['salary_ask_ratio'] = df['expected_salary'] / (df['salary_max_usd'] + 1.0)
    df['apply_decay_14d'] = np.exp(-df['days_to_apply'].clip(lower=0) / 14.0)
    df['apply_decay_30d'] = np.exp(-df['days_to_apply'].clip(lower=0) / 30.0)

    prev = df["previous_companies"].apply(tokenize_pipe)
    if fitted_company_freq is None:
        freq = Counter()
        for s in prev:
            freq.update(s)
    else:
        freq = fitted_company_freq
    df["max_company_freq"] = [max((freq[c] for c in s), default=0) for s in prev]
    df["mean_company_freq"] = [np.mean([freq[c] for c in s]) if s else 0.0 for s in prev]
    df["n_previous_companies"] = prev.apply(len)

    df["completeness_x_skill"] = df["profile_completeness"] * df["skill_jaccard"]
    df["idf_x_inband"] = df["skill_match_idf"] * df["in_experience_band"]
    df["idf_x_salband"] = df["skill_match_idf"] * df["salary_band_distance"]

    g = df.groupby('job_id')
    df["expgrank_x_salgrank"] = g["experience_gap"].rank(pct=True) * g["salary_gap"].rank(pct=True)
    df["title_x_expgrank"] = df["title_token_jaccard"] * g["experience_gap"].rank(pct=True)
    df["match_per_salovershoot"] = df["skill_match_idf"] / (df["salary_band_distance"] + 1.0)
    df["match_per_expgap"] = df["skill_match_idf"] / (df["exp_band_distance"] + 1.0)

    rs = g["skill_match_idf"].rank(ascending=False, method="min")
    re = g["exp_band_distance"].rank(ascending=True, method="min")
    df["consensus_rr"] = 1.0 / (rs + 1.0) + 1.0 / (re + 1.0)
    df["rank_spread"] = (rs - re).abs()

    df["skill_z"] = g["skill_match_idf"].transform(lambda x: (x - x.mean()) / (x.std() + 1e-6))
    df["skill_gap_to_best"] = g["skill_match_idf"].transform("max") - df["skill_match_idf"]
    df["idf_gap_to_best"] = g["skill_match_idf"].transform("max") - df["skill_match_idf"]

    same_loc = (df["candidate_location"].fillna("").astype(str).str.lower() ==
                df["job_location"].fillna("").astype(str).str.lower()).astype(int)
    remote_ok = _truthy(df["remote_allowed"]) if "remote_allowed" in df.columns else pd.Series(False, index=df.index)
    relocate_ok = _truthy(df["willing_to_relocate"]) if "willing_to_relocate" in df.columns else pd.Series(False, index=df.index)
    df["loc_same"] = same_loc
    df["loc_compatible"] = ((same_loc == 1) | remote_ok | relocate_ok).astype(int)
    df["loc_friction"] = ((1 - same_loc) * (~remote_ok).astype(int)).astype(int)

    cols_to_rank = [
        'experience_gap', 'salary_gap', 'skill_match_ratio', 'title_similarity',
        'days_to_apply', 'english_score', 'edu_score', "idf_x_inband", "idf_x_salband",
        "expgrank_x_salgrank", "title_x_expgrank", "match_per_salovershoot", "match_per_expgap",
        "consensus_rr", "skill_gap_to_best", "loc_compatible", 'salary_ask_ratio', 'apply_decay_14d', 'cand_app_count'
    ]
    for c in cols_to_rank:
        if c in df.columns:
            df[f"{c}_grank"] = g[c].rank(pct=True)
            df[f"{c}_gzscore"] = g[c].transform(lambda x: (x - x.mean()) / (x.std() + 1e-6))

    return df, idf, freq

train_df, train_idf, train_company_freq = build_advanced_features(train_df)
test_df, _, _ = build_advanced_features(test_df, fitted_idf=train_idf, fitted_company_freq=train_company_freq)

def add_lsa_and_global_features(train_df, test_df):
    text_cols = [('job_title', 'current_title'), ('required_skills', 'skills')]

    med = train_df['expected_salary'].median()
    train_df['global_salary_ratio'] = train_df['expected_salary'] / (med + 1)
    test_df['global_salary_ratio'] = test_df['expected_salary'] / (med + 1)

    train_df['profile_strength'] = train_df['english_score'] + train_df['edu_score'] + train_df['n_certifications']
    test_df['profile_strength'] = test_df['english_score'] + test_df['edu_score'] + test_df['n_certifications']

    for col_job, col_cand in text_cols:
        tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=5000)
        corpus_train = train_df[col_job].fillna("") + " " + train_df[col_cand].fillna("")
        tfidf.fit(corpus_train)

        job_tr = tfidf.transform(train_df[col_job].fillna(""))
        cand_tr = tfidf.transform(train_df[col_cand].fillna(""))
        job_te = tfidf.transform(test_df[col_job].fillna(""))
        cand_te = tfidf.transform(test_df[col_cand].fillna(""))

        svd = TruncatedSVD(n_components=10, random_state=RNG_SEED)
        job_tr_s = svd.fit_transform(job_tr)
        cand_tr_s = svd.transform(cand_tr)
        job_te_s = svd.transform(job_te)
        cand_te_s = svd.transform(cand_te)

        train_df[f'{col_job}_svd_sim'] = 1 - paired_cosine_distances(job_tr_s, cand_tr_s)
        test_df[f'{col_job}_svd_sim'] = 1 - paired_cosine_distances(job_te_s, cand_te_s)

        prod_tr = job_tr_s * cand_tr_s
        prod_te = job_te_s * cand_te_s
        train_df[f'{col_job}_svd_dotmax'] = prod_tr.max(axis=1)
        train_df[f'{col_job}_svd_dotsum'] = prod_tr.sum(axis=1)
        test_df[f'{col_job}_svd_dotmax'] = prod_te.max(axis=1)
        test_df[f'{col_job}_svd_dotsum'] = prod_te.sum(axis=1)

    return train_df, test_df

train_df, test_df = add_lsa_and_global_features(train_df, test_df)

HIGH_CARD_COLS = ['current_title', 'job_location', 'university', 'industry']
TE_K = 20
TE_NOISE = 0.01
TE_MIN_COUNT = 1

def _norm_cat(series):
    return series.fillna("MISSING").astype(str).str.lower().str.strip()

for c in HIGH_CARD_COLS:
    train_df[c + "_normcat"] = _norm_cat(train_df[c])
    test_df[c + "_normcat"] = _norm_cat(test_df[c])

train_df["_nl_full"] = train_df.groupby("job_id")["relevance_label"].transform(
    lambda x: x.rank(pct=True) if len(x) > 1 else 0.5)
_global_gmean = train_df["_nl_full"].mean()

test_te_maps = {}
test_freq_maps = {}
for c in HIGH_CARD_COLS:
    agg = train_df.groupby(c + "_normcat")["_nl_full"].agg(["mean", "count"])
    agg = agg[agg["count"] >= TE_MIN_COUNT]
    smooth = (agg["mean"] * agg["count"] + _global_gmean * TE_K) / (agg["count"] + TE_K)
    test_te_maps[c] = smooth
    test_freq_maps[c] = train_df[c + "_normcat"].value_counts()

def fit_te_on_fold(fold_train_df, cat_col, gmean):
    agg = fold_train_df.groupby(cat_col + "_normcat")["_nl_fold"].agg(["mean", "count"])
    agg = agg[agg["count"] >= TE_MIN_COUNT]
    return (agg["mean"] * agg["count"] + gmean * TE_K) / (agg["count"] + TE_K)

cols_to_drop = [
    'application_id', 'candidate_id', 'job_id', 'application_date', 'job_posted_date',
    'job_title', 'required_skills', 'salary_currency', 'job_location', 'remote_allowed',
    'company_size', 'industry', 'current_title', 'skills', 'education_level',
    'university', 'previous_companies', 'certifications', 'english_proficiency',
    'candidate_location', 'willing_to_relocate', 'account_created_date', 'relevance_label',
    '_nl_full', '_nl_fold',
] + [c + "_normcat" for c in HIGH_CARD_COLS]

for c in HIGH_CARD_COLS:
    train_df[f"{c}_te"] = np.nan
    train_df[f"{c}_freq"] = 0.0
    test_df[f"{c}_te"] = test_df[c + "_normcat"].map(test_te_maps[c]).fillna(_global_gmean).values
    test_df[f"{c}_freq"] = test_df[c + "_normcat"].map(test_freq_maps[c]).fillna(0).values

candidate_features = [
    c for c in train_df.columns
    if c not in cols_to_drop and train_df[c].dtype in [np.float64, np.float32, np.int64, np.int32]
]

def null_importance_pruner(X, y, groups, features, n_runs=15, pct=75, seed=0):
    sort_idx = np.argsort(groups, kind='stable')
    Xs = X.iloc[sort_idx].reset_index(drop=True)
    ys = y[sort_idx].copy()
    _, counts = np.unique(groups[sort_idx], return_counts=True)

    base = lgb.LGBMRanker(n_estimators=200, learning_rate=0.05, num_leaves=63,
                          random_state=seed, n_jobs=-1)
    base.fit(Xs, ys, group=counts)
    actual = base.booster_.feature_importance("gain")

    null = np.zeros((n_runs, len(features)))
    for r in range(n_runs):
        yp = ys.copy()
        start = 0
        rs = np.random.RandomState(1000 + r)
        for cnt in counts:
            block = yp[start:start + cnt].copy()
            rs.shuffle(block)
            yp[start:start + cnt] = block
            start += cnt
        m = lgb.LGBMRanker(n_estimators=200, learning_rate=0.05, num_leaves=63,
                           random_state=r, n_jobs=-1)
        m.fit(Xs, yp, group=counts)
        null[r] = m.booster_.feature_importance("gain")

    thresh = np.percentile(null, pct, axis=0)
    score = np.log1p(actual) - np.log1p(thresh)
    keep = [f for f, s in zip(features, score) if s > 0]
    if len(keep) < 15:
        order = np.argsort(actual)[::-1]
        keep = [features[i] for i in order[:50]]
    return keep

te_cols = [f"{c}_te" for c in HIGH_CARD_COLS] + [f"{c}_freq" for c in HIGH_CARD_COLS]
static_features = [c for c in candidate_features if c not in te_cols]

kept_static = null_importance_pruner(
    train_df[static_features],
    train_df['relevance_label'].values,
    train_df['job_id'].values,
    static_features,
)
final_features = kept_static + te_cols
print(f"[pruner] kept {len(kept_static)} static + {len(te_cols)} TE = {len(final_features)} features")

train_df = train_df.sort_values('job_id').reset_index(drop=True)
test_df = test_df.sort_values('job_id').reset_index(drop=True)

def ndcg_by_group(y_true, y_pred, groups, k=10):
    d = pd.DataFrame({"t": y_true, "p": y_pred, "g": groups})
    s = [ndcg_score([gr["t"].values], [gr["p"].values], k=k)
         for _, gr in d.groupby("g") if len(gr) >= 2]
    return float(np.mean(s))

def group_rank(pred, groups):
    return pd.Series(pred).groupby(groups).rank(pct=True).values

y = train_df['relevance_label'].values
groups = train_df['job_id'].values

lgb_params = {
    'objective': 'lambdarank',
    'metric': 'ndcg',
    'eval_at': [10],
    'lambdarank_truncation_level': 12,
    'n_estimators': 3000,
    'learning_rate': 0.02,
    'num_leaves': 63,
    'max_depth': 7,
    'min_child_samples': 30,
    'min_split_gain': 0.0,
    'subsample': 0.8,
    'subsample_freq': 1,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.5,
    'reg_lambda': 5.0,
    'label_gain': [0, 1, 3, 7, 15],
    'random_state': RNG_SEED,
    'n_jobs': -1,
    'verbosity': -1,
}

cat_params = dict(
    loss_function="YetiRankPairwise",
    eval_metric="NDCG:top=10",
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=5,
    random_seed=RNG_SEED,
    verbose=0,
)
job_dates = train_df.groupby('job_id')['application_date'].max().sort_values()
time_sorted_jobs = job_dates.index.to_numpy()
folds = np.array_split(time_sorted_jobs, 5)

cv_splits = []
for i in range(1, 5):
    tr_jobs = np.concatenate(folds[:i])
    val_jobs = folds[i]
    tr_idx = train_df.index[train_df['job_id'].isin(tr_jobs)].to_numpy()
    val_idx = train_df.index[train_df['job_id'].isin(val_jobs)].to_numpy()
    cv_splits.append((tr_idx, val_idx))

oof_lgb = np.full(len(train_df), np.nan)
oof_cat = np.full(len(train_df), np.nan)
lgb_best_iters, cat_best_iters = [], []
rng = np.random.default_rng(RNG_SEED)

for fold, (tr_idx, val_idx) in enumerate(cv_splits):
    fold_tr = train_df.iloc[tr_idx].copy()
    fold_tr["_nl_fold"] = fold_tr.groupby("job_id")["relevance_label"].transform(
        lambda x: x.rank(pct=True) if len(x) > 1 else 0.5)
    gmean = fold_tr["_nl_fold"].mean()

    for c in HIGH_CARD_COLS:
        smap = fit_te_on_fold(fold_tr, c, gmean)
        val_vals = train_df.iloc[val_idx][c + "_normcat"].map(smap).fillna(gmean).values
        train_df.loc[train_df.index[val_idx], f"{c}_te"] = val_vals
        tr_vals = train_df.iloc[tr_idx][c + "_normcat"].map(smap).fillna(gmean).values
        tr_vals = tr_vals * (1 + rng.normal(0, TE_NOISE, size=len(tr_vals)))
        train_df.loc[train_df.index[tr_idx], f"{c}_te"] = tr_vals
        freq_map = fold_tr[c + "_normcat"].value_counts()
        train_df.loc[train_df.index[tr_idx], f"{c}_freq"] = \
            train_df.iloc[tr_idx][c + "_normcat"].map(freq_map).fillna(0).values
        train_df.loc[train_df.index[val_idx], f"{c}_freq"] = \
            train_df.iloc[val_idx][c + "_normcat"].map(freq_map).fillna(0).values

    X = train_df[final_features]

    tr_order = np.argsort(groups[tr_idx], kind="stable")
    Xtr = X.iloc[tr_idx[tr_order]]
    ytr = y[tr_idx[tr_order]]
    gtr = groups[tr_idx[tr_order]]

    va_order = np.argsort(groups[val_idx], kind="stable")
    Xva = X.iloc[val_idx[va_order]]
    yva = y[val_idx[va_order]]
    gva = groups[val_idx[va_order]]

    _, c_tr = np.unique(gtr, return_counts=True)
    _, c_va = np.unique(gva, return_counts=True)

    lm = lgb.LGBMRanker(**lgb_params)
    lm.fit(Xtr, ytr, group=c_tr,
           eval_set=[(Xva, yva)], eval_group=[c_va],
           callbacks=[lgb.early_stopping(50, verbose=False)])
    lgb_best_iters.append(lm.best_iteration_ or lgb_params['n_estimators'])
    oof_lgb[val_idx] = lm.predict(X.iloc[val_idx])

    cm = CatBoostRanker(**cat_params, early_stopping_rounds=50)
    cm.fit(Pool(Xtr, ytr, group_id=gtr), eval_set=Pool(Xva, yva, group_id=gva))
    cat_best_iters.append(cm.get_best_iteration() or cat_params['iterations'])
    oof_cat[val_idx] = cm.predict(X.iloc[val_idx])

    f_lgb = ndcg_by_group(y[val_idx], oof_lgb[val_idx], groups[val_idx])
    f_cat = ndcg_by_group(y[val_idx], oof_cat[val_idx], groups[val_idx])
    print(f"[fold {fold}] NDCG@10  lgb={f_lgb:.4f}  cat={f_cat:.4f}  "
          f"(lgb_iter={lgb_best_iters[-1]}, cat_iter={cat_best_iters[-1]})")

valid = ~np.isnan(oof_lgb)
r_lgb = group_rank(oof_lgb, groups)
r_cat = group_rank(oof_cat, groups)

print(f"\n[OOF] lgb={ndcg_by_group(y[valid], oof_lgb[valid], groups[valid]):.4f}  "
      f"cat={ndcg_by_group(y[valid], oof_cat[valid], groups[valid]):.4f}")

def power_blend(rl, rc, w, p):
    return w * (rl ** p) + (1 - w) * (rc ** p)

def best_blend(mask):
    bs, bw, bp = -1.0, 0.5, 1.0
    for p in (1.0, 1.5, 2.0):
        for w in np.linspace(0, 1, 41):
            sc = ndcg_by_group(y[mask], power_blend(r_lgb, r_cat, w, p)[mask], groups[mask])
            if sc > bs:
                bs, bw, bp = sc, w, p
    return bw, bp, bs

w_all, p_all, s_all = best_blend(valid)
print(f"[blend] global  w={w_all:.3f} p={p_all}  NDCG@10={s_all:.4f}")

w_seen, p_seen = w_all, p_all
w_cold, p_cold = max(0.0, w_all - 0.15), p_all

for c in HIGH_CARD_COLS:
    agg = train_df.groupby(c + "_normcat")["_nl_full"].agg(["mean", "count"])
    agg = agg[agg["count"] >= TE_MIN_COUNT]
    smooth = (agg["mean"] * agg["count"] + _global_gmean * TE_K) / (agg["count"] + TE_K)
    train_df[f"{c}_te"] = train_df[c + "_normcat"].map(smooth).fillna(_global_gmean).values
    train_df[f"{c}_freq"] = train_df[c + "_normcat"].map(
        train_df[c + "_normcat"].value_counts()).fillna(0).values

X = train_df[final_features]
_, c_full = np.unique(groups, return_counts=True)
sort_full = np.argsort(groups, kind="stable")

lgb_final_iter = int(np.median(lgb_best_iters))
cat_final_iter = int(np.median(cat_best_iters))
print(f"[final] lgb_iter={lgb_final_iter}  cat_iter={cat_final_iter}")

final_lgb_params = {**lgb_params, 'n_estimators': lgb_final_iter}
final_lgb = lgb.LGBMRanker(**final_lgb_params)
final_lgb.fit(X.iloc[sort_full], y[sort_full], group=c_full)

final_cat_params = {**cat_params, 'iterations': cat_final_iter}
final_cat = CatBoostRanker(**final_cat_params)
final_cat.fit(Pool(X.iloc[sort_full], y[sort_full], group_id=groups[sort_full]))

X_test = test_df[final_features]
test_groups = test_df['job_id'].values

rt_lgb = group_rank(final_lgb.predict(X_test), test_groups)
rt_cat = group_rank(final_cat.predict(X_test), test_groups)

is_cold = ~test_df['job_id'].isin(train_job_ids).values
final_blend = np.empty(len(test_df))
final_blend[~is_cold] = power_blend(rt_lgb, rt_cat, w_seen, p_seen)[~is_cold]
final_blend[is_cold] = power_blend(rt_lgb, rt_cat, w_cold, p_cold)[is_cold]

print(f"[test] cold-start rows: {is_cold.sum()} / {len(is_cold)} "
      f"({100*is_cold.mean():.1f}%)")

sub = pd.DataFrame({'application_id': test_df['application_id'], 'score': final_blend})
sub = sub.sort_values('application_id').reset_index(drop=True)
sub.to_csv('submission_ensemble_final.csv', index=False)
print("[done] wrote submission_ensemble_final.csv")

[pruner] kept 37 static + 8 TE = 45 features
[fold 0] NDCG@10  lgb=0.8652  cat=0.8712  (lgb_iter=543, cat_iter=1990)
[fold 1] NDCG@10  lgb=0.8758  cat=0.8752  (lgb_iter=675, cat_iter=1917)
[fold 2] NDCG@10  lgb=0.8924  cat=0.8855  (lgb_iter=964, cat_iter=1992)
[fold 3] NDCG@10  lgb=0.8801  cat=0.8698  (lgb_iter=1122, cat_iter=1999)

[OOF] lgb=0.8783  cat=0.8754
[blend] global  w=0.625 p=1.0  NDCG@10=0.8789
[final] lgb_iter=819  cat_iter=1991
[test] cold-start rows: 48392 / 52700 (91.8%)
[done] wrote submission_ensemble_final.csv
